In [18]:
from langchain_classic.memory import ConversationBufferMemory, ChatMessageHistory
from langchain_classic.chains import ConversationChain
from langchain_core.messages import HumanMessage, AIMessage
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate

# 1. Set up the language model
# model_id = 'meta-llama/llama-4-maverick-17b-128e-instruct-fp8'
# parameters = {
#     GenParams.MAX_NEW_TOKENS: 256,
#     GenParams.TEMPERATURE: 0.2,
# }
# credentials = {"url": "https://us-south.ml.cloud.ibm.com"}
# project_id = "skills-network"

# Initialize the model
model = "llama3.2:1b"

llm = ChatOllama(
    model=model,
    temperature=0
)

# 2. Create a simple conversation with chat history
history = ChatMessageHistory()

# Add some initial messages (optional)
history.add_user_message("Hello, my name is Alice.")
history.add_ai_message("Hi Alice!")

# 3. Print the current conversation history
# history.messages

# 4. Set up a conversation chain with memory. Pass prompt for better performance
memory = ConversationBufferMemory(
    memory_key="history",
    return_messages=False
)

prompt = PromptTemplate(
    input_variables=["history", "input"],
    template="""
    You are a helpful conversational assistant.
    
    Use the conversation history to answer questions.
    Remember facts the user has shared.
    Respond directly without repeating Human: or AI:
    
    Conversation history: {history}
    
    Human: {input}
    """
)

conversation = ConversationChain(
    llm = llm,
    prompt = prompt,
    verbose = True,
    memory = memory
)

# 5. Function to simulate a conversation
def chat_simulation(conversation, inputs):
    """Run a series of inputs through the conversation chain and display responses"""
    print("\n=== Beginning Chat Simulation ===")
    
    for i, user_input in enumerate(inputs):
        print(f"\n--- Turn {i+1} ---")
        print(f"Human: {user_input}")
        
        # Get response from the conversation chain
        response = conversation.invoke(input=user_input)
        
        # Print the AI's response
        print(f"AI: {response['response']}")
    
    print("\n=== End of Chat Simulation ===")

# 6. Test with a series of related questions
test_inputs = [
    "My favorite color is blue.",
    "I enjoy hiking in the mountains.",
    "What activities would you recommend for me?",
    "What was my favorite color again?",
    "Can you remember both my name and my favorite color?"
]

chat_simulation(conversation, test_inputs)

# 7. Examine the conversation memory
print("\nFinal Memory Contents:")
##TODO: Print the contents of the conversation memory

memory_variables = memory.load_memory_variables({})

print(memory_variables["history"])

# 8. Create a new conversation with a different type of memory (optional)
# Try implementing ConversationSummaryMemory or another type of memory


=== Beginning Chat Simulation ===

--- Turn 1 ---
Human: My favorite color is blue.


> Entering new ConversationChain chain...
Prompt after formatting:

    You are a helpful conversational assistant.

    Use the conversation history to answer questions.
    Remember facts the user has shared.
    Respond directly without repeating Human: or AI:

    Conversation history:
    

    Human: My favorite color is blue.
    

> Finished chain.
AI: Blue is a calming and trustworthy color, often associated with feelings of serenity and tranquility.

--- Turn 2 ---
Human: I enjoy hiking in the mountains.


> Entering new ConversationChain chain...
Prompt after formatting:

    You are a helpful conversational assistant.

    Use the conversation history to answer questions.
    Remember facts the user has shared.
    Respond directly without repeating Human: or AI:

    Conversation history:
    Human: My favorite color is blue.
AI: Blue is a calming and trustworthy color, often associated 

### Using modern **RunnableWithMessageHistory**

In [19]:

from langchain_ollama import ChatOllama

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# 1. Initialize the LLM

model="llama3.2:1b"

llm = ChatOllama(
    model= model,
    temperature=0
)

# 2. Create a chat prompt
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a helpful conversational assistant.
        Use the conversation history to answer questions.
        Do not invent personal information."""
    ),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

# 3. Create the chain
chain = prompt | llm

# 4. Store conversation histories
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()

    return store[session_id]

# 5. Add message history to the chain
conversation = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

# 6. Function to simulate conversation
def chat_simulation(inputs, session_id="user_1"):

    print("\n=== Beginning Chat Simulation ===")

    for i, user_input in enumerate(inputs):

        print(f"\n--- Turn {i+1} ---")
        print(f"Human: {user_input}")

        response = conversation.invoke(
            {"input": user_input},
            config={
                "configurable": {
                    "session_id": session_id
                }
            }
        )

        print(f"AI: {response.content}")

    print("\n=== End of Chat Simulation ===")

# 7. Test conversation
test_inputs = [
    "My name is Alice.",
    "My favorite color is blue.",
    "I enjoy hiking in the mountains.",
    "What activities would you recommend for me?",
    "What was my favorite color again?",
    "Can you remember both my name and my favorite color?"
]

chat_simulation(test_inputs)

# 8. Inspect conversation history
print("\nFinal Memory Contents:")

for message in get_session_history("user_1").messages:
    print(f"{message.type}: {message.content}")


/Users/maleeha/Documents/Learn_AI/langchain-project/venv/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)



=== Beginning Chat Simulation ===

--- Turn 1 ---
Human: My name is Alice.
AI: Hello Alice, how can I help you today?

--- Turn 2 ---
Human: My favorite color is blue.
AI: Blue is a lovely color. Many people find it calming and soothing. Do you have a particular shade of blue that you like the most, or is it more of a general preference?

--- Turn 3 ---
Human: I enjoy hiking in the mountains.
AI: There's something about being in nature, surrounded by mountains and trees, that can be very invigorating. Blue skies and clear mountain air can be especially beautiful. Have you ever hiked to a particularly stunning viewpoint or waterfall in the mountains?

--- Turn 4 ---
Human: What activities would you recommend for me?
AI: Considering your love for hiking and the outdoors, I'd recommend trying some other activities that allow you to connect with nature. You might enjoy:

1. Kayaking or canoeing on a nearby lake or river.
2. Cycling through scenic trails or taking a leisurely ride on a bik